In [2]:
# ============================================================
# Load TCGA-LUAD STAR TPM Gene Expression Matrix from Google Drive
# Folder:
# My Drive/Computational Biology Final Project - Group 9/
# File:
# TCGA-LUAD.star_tpm.tsv.gz
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil
import pandas as pd
import numpy as np

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Define project path
PROJECT_DIR = Path("/content/drive/MyDrive/Computational Biology Final Project - Group 9")
RAW_FILE = PROJECT_DIR / "TCGA-LUAD.star_tpm.tsv.gz"

# 3. Validate file exists
if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"File not found:\n{RAW_FILE}\n\n"
        "Pastikan file TCGA-LUAD.star_tpm.tsv.gz ada di:\n"
        "My Drive -> Computational Biology Final Project - Group 9"
    )

# 4. Copy file to Colab local runtime for faster reading
LOCAL_DIR = Path("/content/tcga_luad")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILE = LOCAL_DIR / "TCGA-LUAD.star_tpm.tsv.gz"

if not LOCAL_FILE.exists():
    shutil.copy2(RAW_FILE, LOCAL_FILE)

print(f"Local file ready: {LOCAL_FILE}")

# 5. Load expression matrix
# Xena expression matrix is usually gene x sample
expr_raw = pd.read_csv(LOCAL_FILE, sep="\t", compression="infer")

print("Raw expression matrix shape:", expr_raw.shape)
display(expr_raw.head())

# 6. Detect gene identifier column
gene_col = expr_raw.columns[0]
print("Detected gene column:", gene_col)

# 7. Convert gene x sample into sample x gene
expr_ml = expr_raw.set_index(gene_col).T.reset_index()
expr_ml = expr_ml.rename(columns={"index": "sample_id"})

# 8. Create Tumor/Normal label from TCGA barcode
# TCGA sample type code:
# 01 = Primary Solid Tumor
# 11 = Solid Tissue Normal
def get_label_from_tcga_barcode(barcode):
    parts = str(barcode).split("-")

    if len(parts) < 4:
        return "Unknown"

    sample_type_code = parts[3][:2]

    if sample_type_code in ["01", "02", "03", "04", "05", "06", "07", "08", "09"]:
        return "Tumor"
    elif sample_type_code in ["10", "11", "12", "13", "14"]:
        return "Normal"
    else:
        return "Unknown"

expr_ml["label"] = expr_ml["sample_id"].apply(get_label_from_tcga_barcode)

# 9. Keep only Tumor and Normal samples
expr_ml = expr_ml[expr_ml["label"].isin(["Tumor", "Normal"])].reset_index(drop=True)

print("\nFinal ML dataset shape:", expr_ml.shape)
print("\nLabel distribution:")
print(expr_ml["label"].value_counts())

display(expr_ml.head())

# 10. Save processed dataset back to Google Drive
PROCESSED_FILE = PROJECT_DIR / "tcga_luad_star_tpm_ml_dataset.parquet"
expr_ml.to_parquet(PROCESSED_FILE, index=False)

print(f"\nProcessed dataset saved to:\n{PROCESSED_FILE}")

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Computational Biology Final Project - Group 9")
PROCESSED_FILE = PROJECT_DIR / "tcga_luad_star_tpm_ml_dataset.parquet"

expr_ml = pd.read_parquet(PROCESSED_FILE)

print(expr_ml.shape)
print(expr_ml["label"].value_counts())
display(expr_ml.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local file ready: /content/tcga_luad/TCGA-LUAD.star_tpm.tsv.gz
Raw expression matrix shape: (60660, 590)


,Ensembl_ID,TCGA-38-7271-01A,TCGA-55-7914-01A,TCGA-95-7043-01A,TCGA-73-4658-01A,TCGA-86-8076-01A,TCGA-55-7726-01A,TCGA-44-6147-01A,TCGA-50-5932-01A,TCGA-44-2661-01A,...,TCGA-50-5946-02A,TCGA-86-7713-01A,TCGA-86-8073-01A,TCGA-44-2662-01B,TCGA-MN-A4N4-01A,TCGA-53-7626-01A,TCGA-62-A46O-01A,TCGA-44-A47G-01A,TCGA-55-6969-01A,TCGA-55-6969-11A
0,ENSG00000000003.15,4.993860,5.572080,5.037615,6.157945,5.061914,5.071063,5.385517,6.225234,5.659973,...,5.877877,6.605066,5.321856,5.100397,5.620695,5.077734,6.845089,5.103242,4.201736,3.721241
1,ENSG00000000005.6,0.000000,0.000000,0.000000,3.598949,0.000000,0.000000,0.196985,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.341758,0.000000,0.000000,0.000000,0.000000,0.000000,0.101650
2,ENSG00000000419.13,5.876966,6.447268,6.706663,6.168664,5.867071,7.343107,6.391027,6.585406,6.432363,...,6.874345,6.771603,6.987923,6.713725,6.928426,6.241032,6.418473,5.584301,6.376573,6.094359
3,ENSG00000000457.14,2.954848,3.269826,2.656977,2.761775,3.112600,2.552623,3.426198,3.774302,3.060912,...,3.099379,4.229765,3.100843,4.618344,3.059217,3.313318,3.158644,2.693632,2.542852,2.696061
4,ENSG00000000460.17,1.858379,1.965212,1.934139,1.726788,1.763412,2.477237,2.423430,2.314232,1.848958,...,3.202151,4.033529,2.616358,4.613820,2.562841,2.095047,3.221460,1.687509,2.272292,1.212756


Detected gene column: Ensembl_ID

Final ML dataset shape: (589, 60662)

Label distribution:
label
Tumor     530
Normal     59
Name: count, dtype: int64


Ensembl_ID,sample_id,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,...,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1,label
0,TCGA-38-7271-01A,4.993860,0.000000,5.876966,2.954848,1.858379,4.675342,5.643836,5.187103,3.253233,...,0.0,0.089498,0.0,0.0,0.032665,3.011066,0.0,0.079702,0.573665,Tumor
1,TCGA-55-7914-01A,5.572080,0.000000,6.447268,3.269826,1.965212,3.052677,3.757482,5.190536,1.529521,...,0.0,0.339251,0.0,0.0,0.000000,4.229549,0.0,0.021480,0.514299,Tumor
2,TCGA-95-7043-01A,5.037615,0.000000,6.706663,2.656977,1.934139,1.742222,3.667631,5.910114,6.857140,...,0.0,0.249749,0.0,0.0,0.000000,2.682483,0.0,0.014641,0.232538,Tumor
3,TCGA-73-4658-01A,6.157945,3.598949,6.168664,2.761775,1.726788,5.441371,5.503400,5.536134,3.210077,...,0.0,0.064745,0.0,0.0,0.000000,2.712508,0.0,0.088820,1.064193,Tumor
4,TCGA-86-8076-01A,5.061914,0.000000,5.867071,3.112600,1.763412,4.725687,6.718786,5.674585,3.744872,...,0.0,0.171463,0.0,0.0,0.000000,3.264416,0.0,0.046421,0.715718,Tumor



Processed dataset saved to:
/content/drive/MyDrive/Computational Biology Final Project - Group 9/tcga_luad_star_tpm_ml_dataset.parquet
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(589, 60662)
label
Tumor     530
Normal     59
Name: count, dtype: int64


,sample_id,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,...,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1,label
0,TCGA-38-7271-01A,4.993860,0.000000,5.876966,2.954848,1.858379,4.675342,5.643836,5.187103,3.253233,...,0.0,0.089498,0.0,0.0,0.032665,3.011066,0.0,0.079702,0.573665,Tumor
1,TCGA-55-7914-01A,5.572080,0.000000,6.447268,3.269826,1.965212,3.052677,3.757482,5.190536,1.529521,...,0.0,0.339251,0.0,0.0,0.000000,4.229549,0.0,0.021480,0.514299,Tumor
2,TCGA-95-7043-01A,5.037615,0.000000,6.706663,2.656977,1.934139,1.742222,3.667631,5.910114,6.857140,...,0.0,0.249749,0.0,0.0,0.000000,2.682483,0.0,0.014641,0.232538,Tumor
3,TCGA-73-4658-01A,6.157945,3.598949,6.168664,2.761775,1.726788,5.441371,5.503400,5.536134,3.210077,...,0.0,0.064745,0.0,0.0,0.000000,2.712508,0.0,0.088820,1.064193,Tumor
4,TCGA-86-8076-01A,5.061914,0.000000,5.867071,3.112600,1.763412,4.725687,6.718786,5.674585,3.744872,...,0.0,0.171463,0.0,0.0,0.000000,3.264416,0.0,0.046421,0.715718,Tumor
